In [ ]:
import sys, json, time
sys.path.insert(0, ".")
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)

from src.config import Config
from src.io import load_raw_data
from src.demand import DemandModel
from src.preprocess import build_candidate_pool, build_prio_lookup
from src.utility import compute_utility_scores, assert_priority_band_separation
from src.routes import build_context, structurally_unreachable, generate_route_pool
from src.qubo import build_route_qubo_full, decode_value
from src.solvers import (solve_exact_bruteforce, simulated_annealing, numpy_qaoa,
                           qiskit_qaoa, build_qaoa_circuit, qiskit_qaoa_xy, build_qaoa_xy_circuit,
                           feasible_mask_all, _all_energies, _all_bits, qiskit_resource_report)
from src.repair import decode_and_repair
from src.outputs import build_todays_orders, build_enriched_inventory, build_route_summary
from src.pipeline import run_pipeline

import os

DATA_DIR = "dataset"      # <-- may be replaced with a different directory on competition day
NUM_VEHICLES = 3           # <-- actual value announced in Appendix A (this competition: 3)
MAX_QUBITS = 19            # <-- qubit budget; the route pool is capped at MAX_QUBITS - NUM_VEHICLES columns
cfg = Config()


## 1. Problem Modeling: Quantum–Classical Decomposition

BNAO requires two simultaneous decisions (Section 1.1): **which ATMs are to be
visited** (selection) and **which vehicle visits them in which order** (routing).
Taken together, these two decisions exhibit a combinatorial explosion (an
independent TSP + knapsack), which makes the problem a natural QUBO candidate.

**Our design decision — route-column encoding:**

- The classical sub-layer (OR-Tools + GRASP/2-opt) first generates a broad pool
  of routes that **already satisfy the 300-minute constraint** (`src/routes.py`).
- The quantum layer operates NOT on the ATMs but on the ROUTES IN THIS POOL,
  with one binary variable $y_r\in\{0,1\}$ per route: *"which $K$ routes
  (K = number of vehicles) should be selected so that the total utility_score
  is maximized, every ATM appears in at most one route, and every mandatory
  ATM appears in exactly one route?"*
- This reduces the qubit count **from the number of candidate ATMs (~40–50)
  to the NUMBER OF ROUTES IN THE POOL (~15–25)** — a decisive difference for
  executability on real quantum hardware (Evaluation Rubric: *Quantum Resource
  Efficiency*). The routes THEMSELVES (the TSP subproblem) remain classical; the
  quantum layer solves only the selection/packing (set-packing) combinatorics
  BETWEEN routes — an approach known in the literature as the combination of
  column generation and QUBO, and widely adopted in quantum approaches to the
  VRP.

Each step below is demonstrated with the same `src/` modules on the ACTUAL
dataset.

## 2. Data Loading and Preprocessing (Sections 1.2–1.3)

In [ ]:
raw = load_raw_data(DATA_DIR)
demand = DemandModel(raw.demand)
task_date = demand.min_date
print("Task date:", task_date.date(), "| CURRENT_INVENTORY:", raw.inventory.shape,
      "| BANKAMATIK_DEMAND:", raw.demand.shape, "| PRIO_LOGIC:", raw.prio.shape)

cand = build_candidate_pool(raw.inventory, raw.prio, demand, task_date, cfg)
print(f"Locations (excluding VEZNE): {len(cand)} | candidates: {cand['is_candidate'].sum()}")
print("Exclusion reasons:", cand.loc[~cand['is_candidate'], 'case_note'].value_counts().to_dict())
print("Candidate case codes:", cand.loc[cand['is_candidate'], 'vaka_kodu_final'].value_counts().to_dict())

scored = compute_utility_scores(cand)
assert_priority_band_separation(scored)   # structural guarantee: priority bands do not overlap
candidates_only = scored[scored["is_candidate"]].reset_index(drop=True)
locationno_by_idx = {i: loc for i, loc in enumerate(candidates_only["locationno"])}
candidates_only.sort_values("utility_score", ascending=False)[
    ["locationno", "vaka_kodu_final", "priority", "mandatory", "ratio", "weight", "utility_score"]].head(10)

## 3. Route Pool: Classical Sub-Layer (Section 1.1 — ≤300 min, Haversine, 35 km/h, no capacity constraint)

In [ ]:
depot = raw.inventory.loc[raw.inventory["type"] == "VEZNE"].iloc[0]
ctx = build_context(candidates_only, float(depot["latitude"]), float(depot["longitude"]), cfg)

unreachable = structurally_unreachable(ctx)
mandatory_idx_all = [i for i, m in enumerate(ctx.mandatory_mask) if m]
mandatory_idx_reachable = [i for i in mandatory_idx_all if i not in unreachable]
print("Locations whose round trip alone exceeds 300 min:", [locationno_by_idx[i] for i in unreachable])
print(f"Mandatory (prio 1-2): {len(mandatory_idx_all)} | reachable: {len(mandatory_idx_reachable)}")

t0 = time.time()
pool, _ = generate_route_pool(ctx, NUM_VEHICLES, pool_size=MAX_QUBITS - NUM_VEHICLES, seed=0)
print(f"Route pool: {len(pool)} feasible routes, {time.time()-t0:.1f}s | source:",
      pd.Series([c['source'] for c in pool]).value_counts().to_dict())
pd.DataFrame(pool).assign(n_stops=lambda d: d["nodes"].apply(len)).sort_values("utility", ascending=False).head(8)

## 4. QUBO Formulation: Route-Column Set Packing with Provably Dominant Penalties

In [ ]:
node_utility = {i: float(ctx.utility[i]) for i in range(len(ctx.utility))}
qp = build_route_qubo_full(pool, NUM_VEHICLES, mandatory_idx_reachable, node_utility)
print(f"Qubits: {qp.Q.shape[0]} (pool + {NUM_VEHICLES} 'empty route' variables)")
print(f"Penalty coefficient A=B=C={qp.A:.2f} (normalized scale; provably dominant via the 4*max_gain+R upper bound)")
print(f"Constant utility contribution of mandatory nodes (removed from the objective): {qp.mandatory_constant:,.1f}")

## 5. Solution: Exact Optimum vs. Simulated Annealing vs. QAOA (Statevector)

In [ ]:
t0 = time.time(); exact = solve_exact_bruteforce(qp, max_R=MAX_QUBITS)
print(f"[Exact optimum] energy={exact['energy']:.6f}  feasible={exact['feasible']}  {time.time()-t0:.1f}s")

t0 = time.time(); sa = simulated_annealing(qp, iters=10_000, restarts=8, seed=0)
print(f"[Simulated Annealing] energy={sa['energy']:.6f}  gap={sa['energy']-exact['energy']:.6f}  {time.time()-t0:.1f}s")

quantum_res = numpy_qaoa(qp, layer_count=2, maxiter=120, n_restarts=3, cvar_alpha=0.02, init="grid", seed=0, max_R=MAX_QUBITS)
assert not quantum_res.get("skipped"), quantum_res.get("reason")
q = quantum_res
print(f"[QAOA p=2, statevector] energy={q['energy']:.6f}  gap={q['energy']-exact['energy']:.6f}  "
      f"{q['runtime_s']:.1f}s  optimizer evaluations={q['optimizer_evals']}")
print(f"  feasible states: {q['num_feasible_states']} / {q['num_states']}  |  feasible mass: {q['feasible_mass']:.4%} "
      f"(uniform distribution {q['uniform_feasible_mass']:.4%}, x{q['feasible_mass']/q['uniform_feasible_mass']:.0f})")
print(f"  P(optimum): {q['p_optimum']:.2e} (rank {q['optimum_rank']})  |  source of the answer: {q['feasible_source']}")

## 6. Qiskit Aer: Shot-Based Execution and Parameter Transfer (Phase 2)

In [ ]:
aer_res = qiskit_qaoa(qp, layer_count=2, theta0=quantum_res["theta"], shots=4000, final_shots=8192,
                      maxiter=40, refine=True, cvar_alpha=0.02, seed=0)
print(f"[Qiskit Aer, {aer_res['final_shots']} shots] gap={aer_res['energy']-exact['energy']:.6f}  "
      f"source={aer_res['feasible_source']}  {aer_res['runtime_s']:.1f}s  |  2-qubit gates: {aer_res['transpiled']['two_qubit_gates']}")
pd.DataFrame({
    "statevector (NumPy)": {"feasible mass": f"{quantum_res['feasible_mass']:.4%}", "P(optimum)": f"{quantum_res['p_optimum']:.2e}",
                            "distance to optimum": round(quantum_res["energy"] - exact["energy"], 6),
                            "theta (γ1, γ2, β1, β2)": np.round(quantum_res["theta"], 3).tolist()},
    "Qiskit Aer (shots)":  {"feasible mass": f"{aer_res['feasible_mass']:.4%}", "P(optimum)": f"{aer_res['p_optimum']:.2e}",
                            "distance to optimum": round(aer_res["energy"] - exact["energy"], 6),
                            "theta (γ1, γ2, β1, β2)": np.round(aer_res["theta"], 3).tolist()},
})

## 7. Hardware-Oriented Formulation: Constraint Preservation with the XY Mixer (Phase 3)

In [ ]:
qp_xy = build_route_qubo_full(pool, NUM_VEHICLES, mandatory_idx_reachable, node_utility, include_cardinality=False)
E_all = _all_energies(qp); fidx = np.where(feasible_mask_all(qp))[0]
worst = int(fidx[np.argmax(E_all[fidx])]); init_bits = _all_bits(qp.Q.shape[0])[worst].astype(int)
print(f"Warm start (worst feasible): energy {E_all[worst]:.4f} | optimum {exact['energy']:.4f} | feasible states {len(fidx)}")

xy_res = qiskit_qaoa_xy(qp_xy, qp, init_bits, layer_count=1, optimize=True, shots=8192, final_shots=8192,
                        cvar_alpha=0.1, grid_gamma=16, grid_beta=10, maxiter=40, seed=0)
print(f"[XY p=1, ideal Aer] (gamma, beta)={np.round(xy_res['theta'], 3).tolist()}  {xy_res['runtime_s']:.0f}s  |  "
      f"remaining in initial state: {xy_res['init_state_fraction']:.2%}")
pd.DataFrame({
    "penalty QUBO, p=2": {"feasible ratio": f"{aer_res['feasible_mass']:.2%}", "K-weight ratio": "—",
                       "better than initial": "—", "P(optimum)": f"{aer_res['p_optimum']:.2%}",
                       "2-qubit gates (all-to-all)": aer_res["transpiled"]["two_qubit_gates"]},
    "XY QUBO, p=1":    {"feasible ratio": f"{xy_res['feasible_mass']:.2%}", "K-weight ratio": f"{xy_res['weight_k_fraction']:.1%}",
                       "better than initial": f"{xy_res['improved_fraction']:.2%}", "P(optimum)": f"{xy_res['p_optimum']:.2%}",
                       "2-qubit gates (all-to-all)": xy_res["transpiled"]["two_qubit_gates"]},
})

### IBM Hardware Results

In [ ]:
try:
    from src.ibm_account import get_runtime_service
    svc, src = get_runtime_service()
    u = svc.usage()
    print(f"quota: {u.get('usage_remaining_seconds')} s remaining / {u.get('usage_limit_seconds')} s")
    rows = []
    for b in svc.backends(simulator=False, operational=True):
        st = b.status(); props = b.properties()
        cz_err = [p.value for g in props.gates if len(g.qubits) == 2 for p in g.parameters if p.name == "gate_error"]
        t1 = [q[0].value for q in props.qubits]
        ro = [p.value for q in props.qubits for p in q if p.name == "readout_error"]
        rows.append({"device": b.name, "qubits": b.num_qubits, "queue": st.pending_jobs,
                     "median CZ error": f"{np.median(cz_err):.2e}" if cz_err else None,
                     "median readout error": f"{np.median(ro):.2e}" if ro else None,
                     "median T1 (us)": round(float(np.median(t1)), 0) if t1 else None,
                     "processor": getattr(getattr(b, "processor_type", None), "get", lambda k, d=None: d)("family", "") if hasattr(b, "processor_type") else ""})
    devices = pd.DataFrame(rows).sort_values(["median CZ error", "queue"]).reset_index(drop=True)
    display(devices)
except Exception as e:
    print(f"Could not retrieve the IBM device list ({type(e).__name__}: {e}). Check the .env file and the internet connection.")


In [ ]:
# ---- OPTIONAL LIVE HARDWARE RUN (default: disabled; consumes quota) ----
RUN_ON_IBM = False
IBM_BACKEND = "ibm_pittsburgh"   # None -> least busy suitable device
IBM_SHOTS = 4000

if RUN_ON_IBM:
    from src.ibm_account import get_runtime_service
    from src.solvers import _runtime_sampler
    svc, _ = get_runtime_service()
    backend = svc.backend(IBM_BACKEND) if IBM_BACKEND else svc.least_busy(min_num_qubits=qp.Q.shape[0], operational=True, simulator=False)
    sampler = _runtime_sampler(backend)
    sampler.options.default_shots = IBM_SHOTS
    sampler.options.dynamical_decoupling.enable = True
    sampler.options.dynamical_decoupling.sequence_type = "XY4"
    sampler.options.twirling.enable_gates = True
    hw_res = qiskit_qaoa_xy(qp_xy, qp, init_bits, layer_count=1, theta0=xy_res["theta"], optimize=False,
                            final_shots=IBM_SHOTS, seed=0, backend=backend, sampler=sampler)
    json.dump({"backend": backend.name, "shots": IBM_SHOTS, "theta": xy_res["theta"],
               "hardware": {k: hw_res[k] for k in ("feasible_mass", "weight_k_fraction", "improved_fraction", "p_optimum", "energy", "transpiled")}},
              open("output/ibm_live_run.json", "w"), indent=1, default=str)
    print(f"[IBM {backend.name}] saved: output/ibm_live_run.json")
else:
    print("RUN_ON_IBM = False -> live hardware run skipped.")

In [ ]:
uniform = len(fidx) / 2 ** qp.Q.shape[0]
def row(r, shots):
    return {"shots": shots, "feasible ratio": f"{r['feasible_mass']:.2%}", "relative to uniform": f"x{r['feasible_mass']/uniform:.0f}",
            "K-weight ratio": f"{r['weight_k_fraction']:.1%}", "better than initial": f"{r['improved_fraction']:.2%}",
            "P(optimum)": f"{r['p_optimum']:.2%}", "CZ": (r.get("transpiled") or {}).get("gate_counts", {}).get("cz") or "—",
            "best feasible gap": None if r.get("energy") is None else round(r["energy"] - exact["energy"], 4)}

rows = {"ideal Aer (this notebook)": row(xy_res, xy_res["final_shots"])}
if os.path.exists("output/ibm_run.json"):
    ibm = json.load(open("output/ibm_run.json"))
    for key, label in (("noisy_aer", f"Aer + {ibm['backend']} noise model"), ("hardware", f"IBM {ibm['backend']} (run_ibm.py)")):
        if ibm.get(key):
            r = dict(ibm[key]); r.setdefault("transpiled", {"gate_counts": ibm["transpiled_hw"]["ops"]})
            rows[label] = row(r, r["final_shots"])
if os.path.exists("output/ibm_live_run.json"):
    live = json.load(open("output/ibm_live_run.json"))
    rows[f"IBM {live['backend']} (live cell)"] = row(live["hardware"], live["shots"])
print(f"Feasible ratio under the uniform distribution: {uniform:.4%}  |  CZ counts refer to the circuit transpiled for the device")
pd.DataFrame(rows).T

## 8. Quantum Resource Usage

In [ ]:
rr = qiskit_resource_report(qp, layer_count=2, heavy_hex_distance=5)
pd.DataFrame({"logical (all-to-all)": {"qubits": rr["logical"]["qubit_count"], "depth": rr["logical"]["depth"], "CX": rr["logical"]["cx_count"]},
              "transpiled to heavy-hex-5": {"qubits": rr["transpiled_heavy_hex"]["physical_qubit_count"], "depth": rr["transpiled_heavy_hex"]["depth"], "CX": rr["transpiled_heavy_hex"]["cx_count"]},
              "XY p=1 (all-to-all, Section 7)": {"qubits": qp.Q.shape[0], "depth": xy_res["transpiled"]["depth"], "CX": xy_res["transpiled"]["two_qubit_gates"]}})

## 9. Repair, Final Route Plan, Order Amounts and Output Files

In [ ]:
if aer_res.get("bits") is not None:        chosen_bits, chosen_src = aer_res["bits"], "qiskit_aer_qaoa"
elif quantum_res.get("bits") is not None:  chosen_bits, chosen_src = quantum_res["bits"], "numpy_qaoa"
else:                                       chosen_bits, chosen_src = exact["bits"], "exact_bruteforce"
repaired = decode_and_repair(qp, ctx, chosen_bits)
print(f"Solution source: {chosen_src} | feasible after repair: {repaired['feasible']} | mandatory nodes that could not be fitted: {repaired['blocked_mandatory']}")

route_minutes = repaired["route_minutes"]
covered_idx = {n for r in repaired["routes"] for n in r}
missed_idx = [i for i in mandatory_idx_all if i not in covered_idx]
missed_locs = [locationno_by_idx[i] for i in missed_idx]
missed_prio1 = [locationno_by_idx[i] for i in missed_idx if candidates_only.iloc[i]["priority"] == 1]
missed_prio2 = [locationno_by_idx[i] for i in missed_idx if candidates_only.iloc[i]["priority"] == 2]
print(f"Missed prio-1: {missed_prio1} | prio-2: {missed_prio2}  (structurally infeasible locations; ASSUMPTIONS.md G)")

madde_by_code = build_prio_lookup(raw.prio)["madde"].to_dict()
todays_orders = build_todays_orders(candidates_only, repaired["routes"], locationno_by_idx, cfg, madde_by_code)
current_inventory = build_enriched_inventory(raw.inventory, scored, repaired["routes"], locationno_by_idx,
                                             route_minutes, missed_locs, missed_prio1, missed_prio2)
route_summary = build_route_summary(repaired["routes"], route_minutes, candidates_only, locationno_by_idx)

todays_orders.to_csv("output/TODAYS_ORDERS.csv", sep="|", index=False)
current_inventory.to_csv("output/CURRENT_INVENTORY.csv", sep="|", index=False)
route_summary.to_csv("output/ROUTE_SUMMARY.csv", sep="|", index=False)
display(route_summary)
todays_orders

## 10. Route Map

Final plan: grey = locations outside the candidate pool, blue = candidates that are not visited, red ring = mandatory (prio 1-2) locations, black star = VEZNE (depot). Each vehicle departs from and returns to the depot, drawn in its own colour.

In [ ]:
import matplotlib.pyplot as plt

depot_lat, depot_lon = float(depot["latitude"]), float(depot["longitude"])
non_cand = scored[~scored["is_candidate"]]
covered = {n for r in repaired["routes"] for n in r}
unvisited = candidates_only.loc[[i for i in range(len(candidates_only)) if i not in covered]]
mandatory_df = candidates_only.loc[candidates_only["mandatory"]]

fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(non_cand["longitude"], non_cand["latitude"], s=18, color="lightgray", label="Not visited / non-candidate")
ax.scatter(unvisited["longitude"], unvisited["latitude"], s=22, color="tab:blue", label="Candidate pool")

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for k, route in enumerate(repaired["routes"]):
    if not route:
        continue
    lats = [depot_lat] + [float(candidates_only.iloc[i]["latitude"]) for i in route] + [depot_lat]
    lons = [depot_lon] + [float(candidates_only.iloc[i]["longitude"]) for i in route] + [depot_lon]
    ax.plot(lons, lats, "-o", color=colors[k % len(colors)], markersize=5, linewidth=1.5,
            label=f"VEHICLE{k + 1} ({len(route)} stops, {route_minutes[k]:.0f} min)")

ax.scatter(mandatory_df["longitude"], mandatory_df["latitude"], s=110, facecolors="none",
           edgecolors="red", linewidths=1.5, label="Mandatory (prio 1-2)")
ax.scatter([depot_lon], [depot_lat], marker="*", s=260, color="black", zorder=5, label="VEZNE (depot)")

ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"Daily route plan — {task_date.date()} | {NUM_VEHICLES} vehicles | solution: {chosen_src}")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("output/route_map.png", dpi=150)
plt.show()